In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
MODE = "light"

In [ ]:
import os
import sys
import django

# Setup Django environment
# Adjust the path to point to the directory containing manage.py
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../hiccup_ide")))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hiccup_ide.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

In [ ]:
from neural_data.models import *
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import (
    show_single_channel_red_green_black as S,
    to_show_list as tsl,
    show_72,
    show_72_list,
    scatter_plot_1d,
    mk_rect_on_ax,
    get_receptive,
    otsu_threshold,
    explain_variance_with_pca,
)
from tqdm import tqdm
from pt_to_api.contribs.v1 import (
    show_input_patch_and_kernel_placement_for_poi_using_raw_params as SIP,
)
import seaborn as sns
import numpy as np
from sklearn.decomposition import MiniBatchDictionaryLearning, DictionaryLearning
from sklearn.preprocessing import Normalizer
from sklearn.metrics.pairwise import (
    pairwise_distances,
    cosine_similarity,
    cosine_distances,
)
from scipy.optimize import linear_sum_assignment
import pandas as pd

from neural_data.utils import (
    get_full_conv_kernel_at_coordinate,
    get_saliency_map_ids_and_patches,
    get_full_activations_of_layer,
)
from collections import defaultdict
import itertools
from torch import nn
from torch import optim


In [ ]:
from torch import nn
from torch import optim


class GroupingAutoencoderFixedSigma(nn.Module):
    def __init__(self, input_dim, n_components):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, n_components, bias=True),
        )
        self.decoder = nn.Linear(n_components, input_dim, bias=False)

    def forward(self, x):
        codes = self.encoder(x)
        recon = self.decoder(codes)
        return recon, codes

    def loss_fn(self, x, recons, codes, alpha, sigma_x, sigma_s, sigma_0):
        recons_loss = self.recon_loss(x, recons, sigma_x)

        codes_loss = self.codes_loss(codes, sigma_s)

        weights_loss = self.weights_loss(alpha, sigma_0)

        return recons_loss, codes_loss, weights_loss

    def codes_loss(self, codes, sigma_s):
        return self.gauss_loss(codes, 0) / (sigma_s * sigma_s)

    def recon_loss(self, x, recons, sigma_x):
        return self.gauss_loss(x, recons) / (sigma_x * sigma_x)

    def cauchy_loss(self, x, cauchy_gamma):
        return torch.log(1 + (x * x) / (cauchy_gamma * cauchy_gamma)).sum()

    def cauchy_codes_loss(self, codes, cauchy_gamma):
        return (
            torch.log(1 + (codes * codes) / (cauchy_gamma * cauchy_gamma)).sum(1).mean()
        )

    def gauss_loss(self, x, mean):
        loss = (x - mean) ** 2
        # sum the loss inside one example, send back mean across examples for the batch
        return torch.sum(loss, 1).mean()

    def l1_loss(self, x, mean):
        loss = (x - mean).abs()
        return torch.sum(loss, 1).mean()

    def weights_loss(self, alpha, sigma_0):
        W = self.decoder.weight
        W_sq = W**2  # (C, K)
        cumsum = torch.cumsum(W_sq, dim=1)  # (C, K), cumsum[c,k] = sum W[c,0..k]^2
        phi = alpha * torch.roll(cumsum, 1, dims=1) + 1  # (C, K)
        phi[:, 0] = 1  # k=0: phi_weight(W, c, -1, alpha) = alpha*0 + 1
        comp1 = (W_sq * phi) / (sigma_0 * sigma_0)
        comp2 = -torch.log(phi)
        return comp1, comp2

    def masked_recon_loss(self, x, recons, sigma_x):
        mask = (x != 0).float()
        loss = ((x - recons) ** 2) * mask
        return torch.sum(loss, 1).mean() / (sigma_x**2)

    def weights_loss_cycled(self, alpha, sigma_0, chunk_size=64):
        W = self.decoder.weight          # (C, K)
        C, K = W.shape

        idx = (torch.arange(K, device=W.device).unsqueeze(0) +
            torch.arange(K, device=W.device).unsqueeze(1)) % K   # (K, K)

        shift_losses = []

        for start in range(0, K, chunk_size):
            idx_chunk = idx[start:start + chunk_size]   # (chunk, K)

            W_chunk = W[:, idx_chunk]                   # (C, chunk, K)
            W_chunk = W_chunk.permute(1, 0, 2)          # (chunk, C, K)

            W_sq = W_chunk ** 2

            cumsum = torch.cumsum(W_sq, dim=2)
            phi = alpha * torch.roll(cumsum, 1, dims=2) + 1
            phi[:, :, 0] = 1

            comp1 = (W_sq * phi) / (sigma_0 * sigma_0)
            comp2 = -torch.log(phi)

            shift_losses.append((comp1 + comp2).sum(dim=(1, 2)))  # (chunk,)

        return torch.cat(shift_losses).mean()

def get_alpha(epoch, total_epochs, alpha_start=0.0, alpha_end=1.0):
    # do the last 25% with max_alpha
    epoch_max = int(total_epochs * 0.30)
    return alpha_start + (alpha_end - alpha_start) * (epoch / epoch_max)


def train_grouping_autoencoder_fixed_sigma(
    X,
    n_components,
    max_alpha=5000,
    sigma_x=0.1,
    sigma_s=1,
    sigma_0=1,
    lr=1e-3,
    epochs=2000,
    batch_size=256,
    weights_algo="cycle",
):
    """
    X: numpy array (n_samples, input_dim)
    n_components: number of dictionary atoms
    alpha: lorentzian sigma shrinker parameter
    sigma_x: std of noise in the data after modelling it as a WS
    sigma_0: std of W, useful to keepn very near 0
    sigma_s: cauchy gamma for cauchy penalty on the encoder (useful for sparse weights)
       the name is sigma_s, cuz it was used as a gaussian prior (L2) on encoder weights, for data which is not sparse
    """
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape

    model = GroupingAutoencoderFixedSigma(input_dim, n_components)
    # torch.nn.init.normal_(model.decoder.weight, 0, 1)

    # svd, might add back again later
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    model.decoder.weight.data = torch.tensor(Vt[:n_components].T, dtype=torch.float32)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        # shuffle
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]

        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]

            # first do on recons, let everything flow, make LR for weight 0
            recon, codes = model(batch)
            recon_loss = model.recon_loss(batch, recon, sigma_x)
            # alpha = get_alpha(epoch, epochs, 100, max_alpha)
            alpha = max_alpha
            # weights_comp1, weights_log_comp = model.weights_loss(alpha, sigma_0)
            # weight_loss = (weights_comp1 + weights_log_comp).sum()
            if weights_algo == "cycled":
                weight_loss = model.weights_loss_cycled(alpha, sigma_0)
            else:
                comp1, comp2 = model.weights_loss(alpha, sigma_0)
                weight_loss = (comp1 + comp2).sum()

            code_loss = model.gauss_loss(model.encoder[0].weight, 0) / (
                sigma_s * sigma_s
            )

            loss = recon_loss + weight_loss + code_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if epoch % 200 == 0:
            msg = f"epoch {epoch:4d} | recon_loss {recon_loss:.4f}"
            if weight_loss is not None:
                # msg += f" weight_comp1 {weights_comp1.sum():.4f} weights_log {weights_log_comp.sum():.4f} codes_loss {code_loss:.4f}"
                msg += f" weight_loss {weight_loss.sum():.4f} codes_loss {code_loss:.4f}"
            # msg += f" reg_weight: {reg_weight}"
            print(msg)

    # final codes and reconstruction
    with torch.no_grad():
        recon, codes = model(X_t)

    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),  # (n_components, input_dim)
        recon.numpy(),
    )


def train_baseline(X, n_components, lr=1e-3, epochs=2000, batch_size=256, sigma_x=1):
    # simple linear model without any non linearities
    # for loss checking
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape

    model = GroupingAutoencoderFixedSigma(input_dim, n_components)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        idx = torch.randperm(n_samples)
        permuted_X_t = X_t[idx]
        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]
            recon, codes = model(batch)
            recon_loss = model.recon_loss(batch, recon, sigma_x)
            loss = recon_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if epoch % 200 == 0:
            print(f"finetune epoch {epoch:4d} | recon_loss {recon_loss:.4f}")

    with torch.no_grad():
        recon, codes = model(X_t)

    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),  # (n_components, input_dim)
        recon.numpy(),
    )


def show_gram(W):
    if not isinstance(W, torch.Tensor):
        W = torch.tensor(W)
    W_norm = W / (W.norm(dim=0, keepdim=True) + 1e-8)
    gram = W_norm.T @ W_norm  # (n_components, n_components)
    plt.imshow(gram, cmap="gray")
    plt.show()
    return gram

# fMRI

https://nilearn.github.io/stable/auto_examples/03_connectivity/plot_compare_decomposition.html#sphx-glr-auto-examples-03-connectivity-plot-compare-decomposition-py

In [ ]:
from nilearn.datasets import fetch_development_fmri

rest_dataset = fetch_development_fmri(n_subjects=30)
func_filenames = rest_dataset.func  # list of 4D nifti files for each subject

# print basic information on the dataset
print(f"First functional nifti image (4D) is at: {rest_dataset.func[0]}")

In [ ]:
from nilearn.image import iter_img
from nilearn.plotting import plot_stat_map, show

def comp_show(components_img, ncols=4):
    images = list(iter_img(components_img))  # your list of stat maps
    nrows = (len(images) + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(20,20))
    axes = axes.flatten()

    for i, (img, ax) in enumerate(zip(images, axes)):
        plot_stat_map(
            img,
            display_mode="z",
            cut_coords=1,
            axes=ax,
            title=f"IC {i}",
            colorbar=False,
            vmax=0.05,
            vmin=-0.05,
        )

    # hide unused axes
    for ax in axes[len(images):]:
        ax.set_visible(False)

    show()

## Fitting ICA, the example in nilearn

In [ ]:
import warnings

from sklearn.exceptions import ConvergenceWarning

from nilearn.decomposition import CanICA

canica = CanICA(
    n_components=20,
    memory="nilearn_cache",
    memory_level=1,
    verbose=1,
    random_state=0,
    mask_strategy="whole-brain-template",
    n_jobs=2,
)
with warnings.catch_warnings():
    # silence warnings about ICA not converging
    # Consider increasing tolerance or the maximum number of iterations.
    warnings.filterwarnings(action="ignore", category=ConvergenceWarning)
    canica.fit(func_filenames)

# Retrieve the independent components in brain space. Directly
# accessible through attribute `components_img_`.
canica_components_img = canica.components_img_
# components_img is a Nifti Image object, and can be saved to a file with
# the following lines:
from pathlib import Path

output_dir = Path.cwd() / "results" / "plot_compare_decomposition"
output_dir.mkdir(exist_ok=True, parents=True)
print(f"Output will be saved to: {output_dir}")
canica_components_img.to_filename(output_dir / "canica_resting_state.nii.gz")

In [ ]:
from nilearn.plotting import plot_prob_atlas

# Plot all ICA components together
plot_prob_atlas(canica_components_img, title="All ICA components")

In [ ]:
comp_show(canica_components_img)

# Dict learning

In [ ]:
from nilearn.decomposition import DictLearning

dict_learning = DictLearning(
    n_components=20,
    memory="nilearn_cache",
    memory_level=1,
    verbose=1,
    random_state=0,
    n_epochs=1,
    mask_strategy="whole-brain-template",
    n_jobs=2,
)

print("[Example] Fitting dictionary learning model")
dict_learning.fit(func_filenames)
print("[Example] Saving results")
# Grab extracted components umasked back to Nifti image.
# Note: For older versions, less than 0.4.1. components_img_
# is not implemented. See Note section above for details.
dictlearning_components_img = dict_learning.components_img_
dictlearning_components_img.to_filename(
    output_dir / "dictionary_learning_resting_state.nii.gz"
)

In [ ]:
plot_prob_atlas(
    dictlearning_components_img, title="All DictLearning components"
)

In [ ]:
images = list(iter_img(dictlearning_components_img))  # your list of stat maps
ncols = 4
nrows = (len(images) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(20,20))
axes = axes.flatten()

for i, (img, ax) in enumerate(zip(images, axes)):
    plot_stat_map(
        img,
        display_mode="z",
        cut_coords=1,
        axes=ax,
        title=f"IC {i}",
        colorbar=False,
        vmax=0.05,
        vmin=-0.05,
    )

# hide unused axes
for ax in axes[len(images):]:
    ax.set_visible(False)

show()

In [ ]:
dict_learning.components_.shape

# our code

In [ ]:
import warnings

import numpy as np
from sklearn.decomposition import dict_learning_online
from sklearn.linear_model import Ridge

from nilearn._utils import logger
from nilearn._utils.docs import fill_doc
from nilearn._utils.helpers import transfer_deprecated_param_vals
from nilearn._utils.param_validation import (
    check_is_of_allowed_type,
    sanitize_verbose,
)
from nilearn.maskers import MultiNiftiMasker, MultiSurfaceMasker
from nilearn.surface import SurfaceImage
from nilearn.typing import NiimgLike

from nilearn.decomposition._base import _BaseDecomposition

class OurModel(_BaseDecomposition):
    def __init__(
        self,
        n_components=20,
        n_epochs=1,
        alpha=10,
        reduction_ratio="auto",
        dict_init=None,
        random_state=None,
        batch_size=20,
        method="cd",
        mask=None,
        smoothing_fwhm=4,
        standardize=True,
        standardize_confounds=True,
        detrend=True,
        low_pass=None,
        high_pass=None,
        t_r=None,
        target_affine=None,
        target_shape=None,
        mask_strategy="epi",
        mask_args=None,
        n_jobs=1,
        verbose=0,
        memory=None,
        memory_level=0,
    ):
        super().__init__(
            n_components=n_components,
            random_state=random_state,
            mask=mask,
            smoothing_fwhm=smoothing_fwhm,
            standardize=standardize,
            standardize_confounds=standardize_confounds,
            detrend=detrend,
            low_pass=low_pass,
            high_pass=high_pass,
            t_r=t_r,
            target_affine=target_affine,
            target_shape=target_shape,
            mask_strategy=mask_strategy,
            mask_args=mask_args,
            memory=memory,
            memory_level=memory_level,
            n_jobs=n_jobs,
            verbose=verbose,
        )
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.method = method
        self.alpha = alpha
        self.reduction_ratio = reduction_ratio
        self.dict_init = dict_init

    def _raw_fit(self, data):
        """Process unmasked data directly.

        Parameters
        ----------
        data : ndarray,
            Shape (n_samples, n_features)

        """
        logger.log("Learning initial components", self.verbose)

        _, n_features = data.shape

        max_iter = ((n_features - 1) // self.batch_size + 1) * self.n_epochs

        std_eps = 1.6939419323878013
        sigma_0 = std_eps*5
        sigma_s = sigma_0*10
        alpha = 5000*(1/(sigma_0*sigma_0))
        model, codes, components, recon = train_grouping_autoencoder_fixed_sigma(
            data.T, self.n_components, epochs=5000, sigma_x=std_eps, sigma_s=sigma_s, sigma_0=sigma_0, max_alpha=alpha
        )
        self.components_ = components
        # self.components_ha
        #     self.n_components,
        #     alpha=self.alpha,
        #     batch_size=self.batch_size,
        #     method=self.method,
        #     dict_init=dict_init,
        #     verbose=max(0, verbose - 1),
        #     random_state=self.random_state,
        #     return_code=True,
        #     shuffle=True,
        #     n_jobs=1,
        #     **kwargs,
        # )
        self.components_ = self.components_.T
        # Unit-variance scaling
        scaled = np.sqrt(np.sum(self.components_**2, axis=1))
        scaled[scaled == 0] = 1
        self.components_ /= scaled[:, np.newaxis]

        # # Flip signs in each component so that positive part is l1 larger
        # # than negative part. Empirically this yield more positive looking maps
        # # than with setting the max to be positive.
        # for component in self.components_:
        #     if np.sum(component > 0) < np.sum(component < 0):
        #         component *= -1
        if hasattr(self, "masker_"):
            self.components_img_ = self.masker_.inverse_transform(
                self.components_
            )

        return self

    def _validate_mask(self):
        if self.mask is not None:
            check_is_of_allowed_type(
                self.mask,
                (
                    MultiSurfaceMasker,
                    SurfaceImage,
                    MultiNiftiMasker,
                    *NiimgLike,
                ),
                "mask",
            )


In [ ]:
dict_learning = OurModel(
    n_components=20,
    memory="nilearn_cache",
    memory_level=1,
    verbose=1,
    random_state=0,
    n_epochs=1,
    mask_strategy="whole-brain-template",
    n_jobs=2,
)

print("[Example] Fitting dictionary learning model")
dict_learning.fit(func_filenames)
print("[Example] Saving results")
# Grab extracted components umasked back to Nifti image.
# Note: For older versions, less than 0.4.1. components_img_
# is not implemented. See Note section above for details.
dictlearning_components_img = dict_learning.components_img_
dictlearning_components_img.to_filename(
    output_dir / "dictionary_learning_resting_state.nii.gz"
)